# Reproducing Tax et al. (2017) — Predictive Business Process Monitoring with LSTM Neural Networks
**Paper:** N. Tax, I. Verenich, M. La Rosa, M. Dumas, CAiSE 2017 — [arXiv:1612.02130](https://arxiv.org/abs/1612.02130)
**Official code:** https://github.com/verenich/ProcessSequencePrediction
**Author of this notebook:** Jules Amimer — MSc AI for Business (NEOMA x INSA)

The notebook runs the **authors' code**, ported from Python 2 / Keras 2.0 to Python 3 / `tf-keras` with minimal changes, on the two logs of the paper, and reproduces **Table 1** (next activity accuracy and next event time MAE) for the configuration published in the repository (LSTM, 2 layers, 1 shared, 100 neurons).

| Log | Paper accuracy | Paper MAE (days) |
|---|---|---|
| Helpdesk | 0.7119 | 3.77 |
| BPI'12 W | 0.7600 | 1.56 |

Runtime: Helpdesk ≈ 15–25 min on CPU. BPI'12 W is much longer (several hours on CPU): use a GPU (*Runtime › Change runtime type › T4 GPU*).

## 1. Setup: official code + my Python 3 port
The port only changes imports, Python 2 syntax and file handling, and passes the log name as an argument. The model, features, split, training and evaluation are unchanged. See `python3_port.diff` in the repository.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # Keras 2 API, as in 2017
!pip install -q tf-keras distance jellyfish
!git clone -q https://github.com/verenich/ProcessSequencePrediction.git
%cd ProcessSequencePrediction/code
for f in ["train.py", "evaluate_next_activity_and_time.py", "evaluate_fixed.py", "summarize_results.py", "baselines_tax_split.py"]:
    !wget -q -O {f} https://raw.githubusercontent.com/jwlamr/lstm-process-monitoring-reproduction/main/code/tax2017/{f}
!rm -f output_files/models/*.h5
!ls

## 2. Data
The logs are in the authors' repository (`data/`), already encoded as integers: `CaseID, ActivityID, CompleteTimestamp`.

In [ ]:
import pandas as pd
for f in ["helpdesk.csv", "bpi_12_w.csv"]:
    d = pd.read_csv("../data/" + f)
    print(f, "cases:", d.CaseID.nunique(), "events:", len(d), "activities:", d.ActivityID.nunique())

## 3. Train on Helpdesk (official script)
Nadam, early stopping (patience 42), up to 500 epochs. The best model is saved in `output_files/models/`.

In [ ]:
!python train.py helpdesk.csv

## 4. Evaluate (official script) and summarise
The script predicts the next activity and its time for every prefix of length 2 ≤ k < case length of the test cases (last third of the log).

In [ ]:
import glob, re
def best_model(log):
    fs = glob.glob(f"output_files/models/{log}_model_*.h5")
    return min(fs, key=lambda f: (float(re.search(r"-([0-9.]+)\.h5", f).group(1)), -int(re.search(r"_(\d+)-", f).group(1))))
m = best_model("helpdesk"); print(m)
!python evaluate_next_activity_and_time.py helpdesk.csv {m}
!python summarize_results.py output_files/results/next_activity_and_time_helpdesk.csv

## 5. Bug check: time features at evaluation
In the official evaluation script, the list of timestamps (`times3`) is never reset between cases, so every test case receives the time-of-day and day-of-week features of the first events of the log. `evaluate_fixed.py` adds the two missing resets; nothing else changes.

In [ ]:
!python evaluate_fixed.py helpdesk.csv {m}
!python summarize_results.py output_files/results/next_activity_and_time_FIXED_helpdesk.csv

## 6. Simple baselines on exactly the same split (my extension)

In [ ]:
!python baselines_tax_split.py ../data/helpdesk.csv

## 7. (Optional, GPU recommended) BPI'12 W

In [ ]:
!python train.py bpi_12_w.csv
m2 = best_model("bpi_12_w")
!python evaluate_next_activity_and_time.py bpi_12_w.csv {m2}
!python summarize_results.py output_files/results/next_activity_and_time_bpi_12_w.csv
!python baselines_tax_split.py ../data/bpi_12_w.csv